In [12]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

StatementMeta(, 4c3d81cc-c9d6-413a-985b-7922f7c03828, 17, Finished, Available, Finished, False)

**đọc bảng bronze**

In [ ]:
df = spark.read.table('resources_lakehouse.bronze.yellow_tripdata')
display(df)

**select + rename + cast data type + filter + drop duplicate**

In [15]:
df_cleaned = df.select( F.col('VendorID').cast('int'),
                        F.col('tpep_pickup_datetime'),
                        F.col('tpep_dropoff_datetime'),
                        F.col('passenger_count'),
                        F.col('trip_distance').cast('double'),
                        F.col('RatecodeID'),
                        F.col('PULocationID'),
                        F.col('DOLocationID'),
                        F.col('fare_amount').cast('double'),
                        F.col('tip_amount').cast('double'),
                        F.col('total_amount').cast('double'),
                        F.col('source_file'),
                        F.col('ingestion_time'))\
                .withColumn('pickup_datetime',F.date_format('tpep_pickup_datetime','yyyy-MM-dd HH:mm:ss').cast('timestamp'))\
                .withColumn('dropoff_datetime',F.date_format('tpep_dropoff_datetime','yyyy-MM-dd HH:mm:ss').cast('timestamp'))\
                .filter((F.col('trip_distance') >0) & (F.col('fare_amount') >0) & (F.col('total_amount') >0) & (F.col('passenger_count') >0)\
                 & (F.col('pickup_datetime') >='2025-01-01 00:00:00') & (F.col('dropoff_datetime') >='2025-01-01 00:00:00') & (F.col('dropoff_datetime') >= F.col('pickup_datetime'))\
                 & (F.col('total_amount') <5000) & (F.col('trip_distance') <500) )\
                .drop('tpep_pickup_datetime','tpep_dropoff_datetime') 

df_silver = df_cleaned.dropDuplicates(['pickup_datetime','dropoff_datetime','PULocationID','DOLocationID','VendorID'])\
                        .withColumn('year',F.year('pickup_datetime'))\
                        .withColumn('month',F.month('pickup_datetime'))\
                        .withColumn('duration_min',((F.unix_timestamp('dropoff_datetime') - F.unix_timestamp('pickup_datetime'))/60).cast('double'))

StatementMeta(, 4c3d81cc-c9d6-413a-985b-7922f7c03828, 20, Finished, Available, Finished, False)

**write to silver table**

In [16]:
df_silver.write.partitionBy('year','month').mode('overwrite').format('delta').option('overwriteSchema','true').saveAsTable('silver.yellow_taxi_tripdata')

StatementMeta(, 4c3d81cc-c9d6-413a-985b-7922f7c03828, 21, Finished, Available, Finished, False)

In [ ]:
df_silver.count()